#Scaling di Features non categoriche

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
# df_path = r"imdb.csv"
df = pd.read_csv("/content/df_completo.csv")

In [ ]:
df.shape

(146332, 61)

In [ ]:
df_nan_runtime = df[df['runtimeMinutes'].isna()].copy()

In [ ]:
df_runtime = df.dropna(subset=['runtimeMinutes'])

In [ ]:
# Divisione train/test
X = df_runtime.drop(columns= ['originalTitle','runtimeMinutes'], axis=1)
y = df_runtime[['runtimeMinutes']]

In [ ]:

features_to_scale = [
    'rating', 'startYear', 'numVotes', 'totalCredits',
    'criticReviewsTotal', 'numRegions', 'userReviewsTotal', 'companiesNumber',
    'averageRating', 'externalLinks', 'writerCredits', 'directorsCredits',
    'quotesTotal', 'totalMedia', 'totalNominations'
]

# Tutte le altre colonne (che non devono essere scalate)
features_not_scaled = [col for col in X.columns if col not in features_to_scale]

In [ ]:
# Inizializza lo scaler
scaler = StandardScaler()

# Applica lo scaling solo sulle colonne selezionate
X_scaled_part = pd.DataFrame(
    scaler.fit_transform(X[features_to_scale]),
    columns=features_to_scale,
    index=X.index
)

# Prendi le altre colonne così come sono
X_not_scaled_part = X[features_not_scaled]


In [ ]:
X_prepared = pd.concat([X_scaled_part, X_not_scaled_part], axis=1)

# Ordina le colonne per sicurezza (opzionale)
X_prepared = X_prepared[X.columns]  # per mantenere l'ordine originale


#Training senza CountryOfOrigin

In [ ]:
# Divisione train/test
X = df_runtime.drop(columns= ['originalTitle','runtimeMinutes', "from_Africa","from_Asia","from_Europe","from_North America","from_Oceania","from_South America"], axis=1)
y = df_runtime[['runtimeMinutes']]

In [ ]:

features_to_scale = [
    'rating', 'startYear', 'numVotes', 'totalCredits',
    'criticReviewsTotal', 'numRegions', 'userReviewsTotal', 'companiesNumber',
    'averageRating', 'externalLinks', 'writerCredits', 'directorsCredits',
    'quotesTotal', 'totalMedia', 'totalNominations'
]

# Tutte le altre colonne (che non devono essere scalate)
features_not_scaled = [col for col in X.columns if col not in features_to_scale]

In [ ]:
# Inizializza lo scaler
scaler = StandardScaler()

# Applica lo scaling solo sulle colonne selezionate
X_scaled_part = pd.DataFrame(
    scaler.fit_transform(X[features_to_scale]),
    columns=features_to_scale,
    index=X.index
)

# Prendi le altre colonne così come sono
X_not_scaled_part = X[features_not_scaled]


In [ ]:
X_prepared = pd.concat([X_scaled_part, X_not_scaled_part], axis=1)

# Ordina le colonne per sicurezza (opzionale)
X_prepared = X_prepared[X.columns]  # per mantenere l'ordine originale


#1% OUTLIER CON IQR SU TITLE TYPE

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Rimuove righe con runtimeMinutes mancanti
#df_runtime = df.dropna(subset=['runtimeMinutes'])

# Lista delle colonne one-hot per titleType
title_type_cols = [
    'titleType_movie', 'titleType_short', 'titleType_tvEpisode',
    'titleType_tvMiniSeries', 'titleType_tvMovie', 'titleType_tvSeries',
    'titleType_tvShort', 'titleType_tvSpecial', 'titleType_video',
    'titleType_videoGame'
]

# Applica filtro IQR su runtimeMinutes separatamente per ciascun titleType
df_list = []
for col in title_type_cols:
    subset = df_runtime[df_runtime[col] == 1]
    #if len(subset) < 100:
     #   continue  # evita categorie troppo piccole
    q1 = subset['runtimeMinutes'].quantile(0.20)
    q3 = subset['runtimeMinutes'].quantile(0.80)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    cleaned = subset[
        (subset['runtimeMinutes'] >= lower_bound) &
        (subset['runtimeMinutes'] <= upper_bound)
    ]
    df_list.append(cleaned)

# Concatenazione dei sotto-dataframe puliti
df_clean = pd.concat(df_list, axis=0)

# Divisione X / y
X = df_clean.drop(columns=[
    'originalTitle', 'runtimeMinutes',
])
y = df_clean[['runtimeMinutes']]

# Feature da scalare
features_to_scale = [
    'rating', 'startYear', 'numVotes', 'totalCredits',
    'criticReviewsTotal', 'numRegions', 'userReviewsTotal', 'companiesNumber',
    'averageRating', 'externalLinks', 'writerCredits', 'directorsCredits',
    'quotesTotal', 'totalMedia', 'totalNominations'
]

# Feature da non scalare
features_not_scaled = [col for col in X.columns if col not in features_to_scale]

# StandardScaler applicato solo a feature selezionate
scaler = StandardScaler()
X_scaled_part = pd.DataFrame(
    scaler.fit_transform(X[features_to_scale]),
    columns=features_to_scale,
    index=X.index
)
X_not_scaled_part = X[features_not_scaled]

# Ricomponi il dataframe completo
X_prepared = pd.concat([X_scaled_part, X_not_scaled_part], axis=1)
X_prepared = X_prepared[X.columns]  # Mantiene ordine originale

# Report finale
print(f"Istanze originali: {len(df_runtime)}")
print(f"Istanze dopo rimozione outlier con IQR per titleType: {len(df_clean)}")


Istanze originali: 107827
Istanze dopo rimozione outlier con IQR per titleType: 105254


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

 #1. Split dei dati usando X_prepared
X_train, X_test, y_train, y_test = train_test_split(
    X_prepared, y, test_size=0.3, random_state=42
)

# 2. Inizializza e addestra il modello
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train.values.ravel())  # .ravel() per passare un array 1D

# 3. Previsioni
y_pred = rf.predict(X_test)

# 4. Valutazione
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# 5. Stampa metriche
print(f"🔍 Mean Squared Error (MSE): {mse:.2f}")
print(f"📉 Mean Absolute Error (MAE): {mae:.2f}")
print(f"📈 R-squared (R²): {r2:.4f}")

🔍 Mean Squared Error (MSE): 202.32
📉 Mean Absolute Error (MAE): 8.39
📈 R-squared (R²): 0.8387


#Regressione su valori NaN del df+ unione dei dataset e df finale

In [ ]:
# 1. Prepara X per le righe con runtimeMinutes mancanti
X_missing = df_nan_runtime[X.columns].copy()

X_missing_scaled_part = pd.DataFrame(
    scaler.transform(X_missing[features_to_scale]),
    columns=features_to_scale,
    index=X_missing.index
)
X_missing_not_scaled_part = X_missing[features_not_scaled]
X_missing_prepared = pd.concat([X_missing_scaled_part, X_missing_not_scaled_part], axis=1)
X_missing_prepared = X_missing_prepared[X.columns]

# 2. Predizione
runtime_predicted = rf.predict(X_missing_prepared)

# 3. Arrotondamento personalizzato
runtime_predicted = np.where(
    runtime_predicted % 1 > 0.59,
    np.ceil(runtime_predicted),
    np.floor(runtime_predicted)
).astype(int)

# 4. Assegna i valori predetti
df_nan_runtime['runtimeMinutes'] = runtime_predicted

# 5. Aggiorna il dataframe originale
df.update(df_nan_runtime[['runtimeMinutes']])


In [ ]:
df_nan_runtime.shape

(38505, 61)

In [ ]:
df_runtime.shape

(107827, 61)

In [ ]:
df.shape

(146332, 61)

In [ ]:
df_nan_runtime

,originalTitle,rating,startYear,runtimeMinutes,numVotes,totalCredits,criticReviewsTotal,numRegions,userReviewsTotal,companiesNumber,...,titleType_movie,titleType_short,titleType_tvEpisode,titleType_tvMiniSeries,titleType_tvMovie,titleType_tvSeries,titleType_tvShort,titleType_tvSpecial,titleType_video,titleType_videoGame
10,Bateau-mouche sur la Seine,4,1896,1,39,2,0,2,0,1,...,0,1,0,0,0,0,0,0,0,0
12,Bois de Boulogne,4,1896,1,38,2,0,2,0,1,...,0,1,0,0,0,0,0,0,0,0
13,Cortège de tzar allant à Versailles,4,1896,1,36,3,0,2,0,0,...,0,1,0,0,0,0,0,0,0,0
14,Cortège de tzar au Bois de Boulogne,4,1896,1,34,2,0,2,0,0,...,0,1,0,0,0,0,0,0,0,0
16,Dessinateur: Chamberlain,3,1896,1,30,2,0,1,0,1,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146319,"Actually, Thank You",8,2019,21,30,27,0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
146321,Ao-chan Can't Easily Decline,7,2019,19,25,58,0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
146323,Poem 4: The Canvas Girl,7,2019,22,12,18,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
146327,Women Take Center Stage,6,2019,39,12,32,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [ ]:
df[df['originalTitle'] == 'Women Take Center Stage']


,originalTitle,rating,startYear,runtimeMinutes,numVotes,totalCredits,criticReviewsTotal,numRegions,userReviewsTotal,companiesNumber,...,titleType_movie,titleType_short,titleType_tvEpisode,titleType_tvMiniSeries,titleType_tvMovie,titleType_tvSeries,titleType_tvShort,titleType_tvSpecial,titleType_video,titleType_videoGame
146327,Women Take Center Stage,6,2019,39.0,12,32,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [ ]:
# Salva il DataFrame su disco
df.to_csv("df_definitivo.csv", index=False)

# Scarica il file su dispositivo locale
from google.colab import files
files.download("df_definitivo.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>